# MedVision Agent — Pancreatic Tumor U-Net (MSD Task07)

> ⚠️ **This is a research prototype and not approved for clinical diagnosis.**
> Use public datasets only and respect their licenses.

**Goal:** train a tiny 2-D U-Net on abdominal CT slices to segment pancreatic
**tumor** pixels (label 2 in MSD Task07; label 1 is the pancreas organ itself),
then export `pancreas_unet.onnx`.

- Dataset: **Medical Segmentation Decathlon — Task07 Pancreas**
  http://medicaldecathlon.com/ (CC-BY-SA; review before redistribution).
  Also available on Kaggle: search for `msd task07 pancreas`.
- Extreme class imbalance: tumor pixels are far below 1% of each slice —
  the Dice term in the loss matters a lot here.
- HU windowing matches the inference service: clip `[-160, 240]`.

In [ ]:
%pip install -q nibabel onnx onnxruntime matplotlib tqdm requests
# torch + torchvision preinstalled on Colab/Kaggle GPUs

In [ ]:
import json, random, shutil, tarfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CONFIG = {
    'SIZE': 256,
    'BATCH': 16,
    'EPOCHS': 30,
    'LR': 1e-3,
    'POS_FRAC': 0.6,     # keep this share of slices tumor-bearing
    'MAX_CASES': 40,
    'HU_CLIP': (-160.0, 240.0),
    'SEED': 42,
}
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
print('device:', DEVICE)

## 1 · Locate MSD Task07 Pancreas data

Expected layout:
```
DATA_ROOT/
  Task07_Pancreas/
    imagesTr/      # NIfTI volumes (.nii.gz)
    labelsTr/      # NIfTI label volumes (.nii.gz)
    dataset.json   # metadata
```

**Download options:**
1. Official mirror: `https://msd-for-monai.s3-us-west-2.amazonaws.com/Task07_Pancreas.tar`
2. Kaggle: search `msd task07 pancreas` and attach as input.
3. The cell below auto-downloads from the S3 mirror if the data isn't found.

In [ ]:
# TODO(student): if your platform blocks this URL, download manually and set DATA_ROOT.
CANDIDATES = [
    '/kaggle/input/msd-task07-pancreas/Task07_Pancreas',
    '/kaggle/input/task07-pancreas/Task07_Pancreas',
    '/content/Task07_Pancreas',
    './data/Task07_Pancreas',
    'data/Task07_Pancreas',
]

DATA_ROOT = next((Path(p) for p in CANDIDATES if Path(p).exists()), None)

if DATA_ROOT is None:
    # Auto-download from the official S3 mirror
    URL = 'https://msd-for-monai.s3-us-west-2.amazonaws.com/Task07_Pancreas.tar'
    TAR_PATH = Path('data/Task07_Pancreas.tar')
    DATA_ROOT = Path('data/Task07_Pancreas')

    TAR_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not TAR_PATH.exists():
        import requests
        print(f'Downloading from {URL} ...')
        with requests.get(URL, stream=True, timeout=60) as r:
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            done = 0
            with open(TAR_PATH, 'wb') as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk); done += len(chunk)
                    if total and done % (100 << 20) < (1 << 20):
                        print(f'{done/1e6:.0f}/{total/1e6:.0f} MB')
    print('extracting...')
    with tarfile.open(TAR_PATH) as tf:
        tf.extractall('data')

print('ready:', DATA_ROOT.exists())

image_paths = sorted((DATA_ROOT / 'imagesTr').glob('*.nii.gz'))
label_paths = sorted((DATA_ROOT / 'labelsTr').glob('*.nii.gz'))
random.Random(CONFIG['SEED']).shuffle(image_paths)
image_paths = image_paths[: CONFIG['MAX_CASES']]
print(f'{len(image_paths)} training cases')

In [ ]:
import nibabel as nib

HU_LO, HU_HI = CONFIG['HU_CLIP']

def window_hu(x):
    x = np.clip(x.astype(np.float32), HU_LO, HU_HI)
    return (x - HU_LO) / (HU_HI - HU_LO)

class PancreasSliceDataset(Dataset):
    '''Axial slices; mask = pancreatic TUMOR only (MSD label == 2).'''

    def __init__(self, image_files, size=256, pos_frac=0.6, augment=False):
        self.size, self.augment = size, augment
        self._cache = {}
        self.index = []  # (img_path, z, has_tumor)
        for ip in image_files:
            lp = Path(str(ip).replace('imagesTr', 'labelsTr'))
            seg = np.asanyarray(nib.load(str(lp)).dataobj)
            pos = sorted(int(z) for z in np.unique(np.nonzero(seg == 2)[2]))
            neg_all = [z for z in range(seg.shape[2]) if z not in set(pos)]
            rng = random.Random((hash(str(ip)) ^ 0xC07A) & 0xFFFF)
            k_neg = min(int(len(pos) * (1 - pos_frac) / max(pos_frac, 1e-6)), len(neg_all))
            for z in pos:
                self.index.append((ip, z, True))
            for z in rng.sample(neg_all, k_neg):
                self.index.append((ip, z, False))

    def _load(self, ip):
        if ip not in self._cache:
            lp = Path(str(ip).replace('imagesTr', 'labelsTr'))
            img = np.asanyarray(nib.load(str(ip)).dataobj, dtype=np.float32)
            msk = (np.asanyarray(nib.load(str(lp)).dataobj) == 2).astype(np.float32)
            self._cache[ip] = (img, msk)  # RAM-heavy; lower MAX_CASES if OOM
        return self._cache[ip]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        ip, z, _ = self.index[i]
        img, msk = self._load(ip)
        x, y = window_hu(img[:, :, z]), msk[:, :, z]

        if self.augment:
            if random.random() < 0.5:
                x, y = x[:, ::-1].copy(), y[:, ::-1].copy()
            k = random.randint(0, 3)
            if k:
                x, y = np.rot90(x, k).copy(), np.rot90(y, k).copy()
            x = x * np.random.uniform(0.92, 1.08)

        rs = (self.size, self.size)
        x = np.asarray(Image.fromarray((np.clip(x, 0, 1) * 255).astype(np.uint8), 'L')
                       .resize(rs, Image.BILINEAR), dtype=np.float32) / 255.0
        y = np.asarray(Image.fromarray((y * 255).astype(np.uint8), 'L')
                       .resize(rs, Image.NEAREST), dtype=np.float32)
        return torch.from_numpy(x[None]).float(), (y > 0.5).float()[None]

split = int(len(image_paths) * 0.85)
train_ds = PancreasSliceDataset(image_paths[:split], CONFIG['SIZE'], CONFIG['POS_FRAC'], augment=True)
val_ds   = PancreasSliceDataset(image_paths[split:], CONFIG['SIZE'], 1.0, augment=False)
train_loader = DataLoader(train_ds, batch_size=CONFIG['BATCH'], shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CONFIG['BATCH'], shuffle=False, num_workers=2, pin_memory=True)
pos_share = sum(1 for _,_,p in train_ds.index if p) / max(len(train_ds), 1)
print(f'train slices {len(train_ds)} ({pos_share:.0%} tumor-bearing) | val cases {split and len(image_paths)-split}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for j in range(4):
    xi = next(i for i in random.sample(range(len(train_ds)), len(train_ds)) if train_ds.index[i][2])
    x, y = train_ds[xi]
    axes[0, j].imshow(x[0], cmap='gray'); axes[0, j].axis('off')
    axes[1, j].imshow(x[0], cmap='gray'); axes[1, j].imshow(y[0], alpha=0.45, cmap='Reds'); axes[1, j].axis('off')
plt.tight_layout(); plt.show()

## 2 · Model, loss, metrics

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class MiniUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.d1, self.d2, self.d3 = DoubleConv(1, base), DoubleConv(base, base*2), DoubleConv(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.u2, self.u1 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2), nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.c2, self.c1 = DoubleConv(base*4, base*2), DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        s1 = self.d1(x); s2 = self.d2(self.pool(s1)); b = self.d3(self.pool(s2))
        u = self.c2(torch.cat([self.u2(b), s2], dim=1))
        u = self.c1(torch.cat([self.u1(u), s1], dim=1))
        return self.out(u)

model = MiniUNet().to(DEVICE)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters')

class DiceBCELoss(nn.Module):
    '''Dice dominates here: tumors are tiny and BCE alone collapses to background.'''
    def __init__(self, dice_weight=0.7, smooth=1.0):
        super().__init__()
        self.bce, self.w, self.smooth = nn.BCEWithLogitsLoss(), dice_weight, smooth
    def forward(self, logits, target):
        bce = self.bce(logits, target)
        prob = torch.sigmoid(logits)
        inter = (prob * target).sum(dim=(2, 3))
        union = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()
        return bce + self.w * dice

@torch.no_grad()
def segmentation_metrics(logits, target, thr=0.5, eps=1e-7):
    pred = (torch.sigmoid(logits) > thr).float()
    tp = (pred * target).sum(); fp = (pred * (1 - target)).sum(); fn = ((1 - pred) * target).sum()
    return {
        'dice': float((2 * tp + eps) / (2 * tp + fp + fn + eps)),
        'iou': float((tp + eps) / (tp + fp + fn + eps)),
        'precision': float((tp + eps) / (tp + fp + eps)),
        'recall': float((tp + eps) / (tp + fn + eps)),
    }

criterion = DiceBCELoss(dice_weight=0.7)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['LR'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

In [ ]:
def train_one_epoch(loader):
    model.train(); total = 0.0
    for img, msk in loader:
        img, msk = img.to(DEVICE), msk.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            loss = criterion(model(img), msk)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total += float(loss) * img.size(0)
    return total / max(1, len(loader.dataset))

@torch.no_grad()
def evaluate(loader):
    model.eval(); total = 0.0; agg = {'dice': [], 'iou': [], 'precision': [], 'recall': []}
    for img, msk in loader:
        img, msk = img.to(DEVICE), msk.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            logits = model(img)
        total += float(criterion(logits.float(), msk)) * img.size(0)
        for k, v in segmentation_metrics(logits.float(), msk).items():
            agg[k].append(v)
    out = {k: sum(v) / len(v) for k, v in agg.items()}
    out['loss'] = total / max(1, len(loader.dataset))
    return out

history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}
best_dice, best_path = -1.0, OUT_DIR / 'pancreas_unet_best.pth'

for epoch in range(1, CONFIG['EPOCHS'] + 1):
    tl = train_one_epoch(train_loader)
    ev = evaluate(val_loader)
    scheduler.step(ev['dice'])
    history['train_loss'].append(tl); history['val_loss'].append(ev['loss'])
    history['val_dice'].append(ev['dice']); history['val_iou'].append(ev['iou'])
    star = ''
    if ev['dice'] > best_dice:
        best_dice = ev['dice']; torch.save(model.state_dict(), best_path); star = '  ← saved'
    print(f"epoch {epoch:02d} | train_loss {tl:.4f} | val_loss {ev['loss']:.4f} | "
          f"dice {ev['dice']:.4f} | iou {ev['iou']:.4f}{star}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['val_loss'], label='val')
ax[0].set_title('Dice+BCE loss'); ax[0].legend()
ax[1].plot(history['val_dice'], label='dice'); ax[1].plot(history['val_iou'], label='iou')
ax[1].set_title('validation overlap metrics'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
final = evaluate(val_loader)
print(json.dumps(final, indent=2))

onnx_path = OUT_DIR / 'pancreas_unet.onnx'
dummy = torch.randn(1, 1, CONFIG['SIZE'], CONFIG['SIZE'])
model.eval().cpu()
torch.onnx.export(
    model, dummy, str(onnx_path), opset_version=17,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch', 2: 'h', 3: 'w'}, 'logits': {0: 'batch', 2: 'h', 3: 'w'}},
)

import onnx, onnxruntime as ort
onnx.checker.check_model(onnx.load(str(onnx_path)))
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
with torch.no_grad():
    ref = torch.sigmoid(model(dummy)).numpy()
raw = sess.run(None, {'input': dummy.numpy()})[0]
prob = raw if (raw.min() >= 0 and raw.max() <= 1) else 1 / (1 + np.exp(-raw))
diff = float(np.abs(prob - ref).max())
print(f'ONNX vs PyTorch max |Δ|: {diff:.2e}')
assert diff < 1e-3, 'export parity check failed'
print(f'Done → copy {onnx_path} to inference-service/models/pancreas_unet.onnx')

## Next steps & honest caveats

- Expect modest Dice on this task even with good pipelines — pancreatic tumor
  segmentation is one of the hardest MSD benchmarks.
- Consider organ-conditioned two-stage models or MONAI's transformer baselines
  when you outgrow this teaching template.
- Copy `outputs/pancreas_unet.onnx` → `inference-service/models/`, then set
  `MOCK_PREDICTIONS=0`.

> **This is a research prototype and not approved for clinical diagnosis.**